# 10_data: OpenAlex pull and label join (CPU runtime)

Thin driver for tasks S1.3 (choose the slice), S1.2 (pilot pull and label join) and S4.0
(full-slice pull). All logic lives in the `citecurv` package; this notebook sets parameters,
calls it, and writes results to Drive.

**PROVISIONAL(G3):** positive labels follow the Referencing/Attributions rule (L12) until the
advisors sign off. Labels come from the frozen Retraction Watch snapshot of 2026-07-06, which is
"publicly available via Crossref".

Before running:
1. Create a free OpenAlex key at openalex.org/settings/api-key and add it as the Colab secret
   `OPENALEX_API_KEY` (left sidebar, key icon). The key is what OpenAlex meters; `mailto` is ignored.
2. Set `TAG`, `STAGE` and `RUN` in the next cell. Run top to bottom; cells for other stages skip themselves.

Budget: list and group-by calls cost 10 credits each; a free key has 100,000 credits a day
(10,000 calls) and 30 requests a second. S1.3 needs roughly 300 to 700 calls; a 300K-work pull
needs about 3,000 pages at 100 works a page.

Drive layout (docs/colab_runbook.md section 5): every artifact of one run lives in
`MyDrive/citecurv/<RUN>/` with `RUN` = `pilot` (S1.2) or `full` (S4.0), under the contract file names.
Set `RUN` in the parameters cell; every notebook (10 to 50) must use the same value.
The pull stage writes `works.parquet`, `edges.parquet`, `events.parquet` (`event_id` assigned once, in
`citecurv.data.events.build_events`), `labels.parquet` and `event_labels.parquet` there; S1.3 files and the
Retraction Watch snapshot stay in `MyDrive/citecurv/` because they are shared by both runs.

In [ ]:
# ---- runtime versions (Colab's default Python changes; citecurv needs >= 3.12) ----
import platform, sys
print("python", sys.version.split()[0], "|", platform.platform())
assert sys.version_info >= (3, 12), "citecurv requires Python 3.12 or newer"

In [ ]:
# ---- parameters -------------------------------------------------------------
REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
TAG = "<release-tag>"          # a tag or branch of REPO; for S4.0 it must contain configs/slice.yaml
STAGE = "S1.3"                 # "S1.3" choose slice | "S1.2" pilot pull + join | "S4.0" full slice
RUN = "pilot"                  # Drive run folder: "pilot" for S1.2, "full" for S4.0 (same value in 20-50)
DRIVE_ROOT = "/content/drive/MyDrive"

RWDB_COMMIT = "21d7124df178ecdd920d05b585f0cae26267aead"   # upstream commit byte-identical to the frozen file
LABEL_FAMILIES = ["referencing/attributions"]              # L12 primary family
LABEL_NATURES = ["Retraction"]                             # decision pending: see docs/slice_shortlist.md
LABEL_JOIN_MODE = "strict"                                 # "strict" | "loose" DOI join for event labels (S1.2 records both)

TARGET_WORKS = 300_000         # S1.3: L11 size target; topics are added until this is reached
CHOSEN = None                  # S1.3: name of the narrowed candidate to write to slice.yaml, e.g. "computing_topics"

PILOT_CANDIDATE = "computing"  # S1.2: SHORTLIST entry that seeds the pilot filter
PILOT_YEAR = 2022
PILOT_MAX_WORKS = 10_000

assert STAGE == "S1.3" or (STAGE, RUN) in {("S1.2", "pilot"), ("S4.0", "full")}, "RUN must be 'pilot' for S1.2 and 'full' for S4.0"

In [ ]:
# ---- install the package and the data-stage pins from the repo tag ------------
import subprocess

assert TAG != "<release-tag>", "set TAG to the release tag first"
subprocess.run(["git", "clone", "--depth", "1", "--branch", TAG, REPO, "/content/repo"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/repo",
                "pyalex==0.21", "pandas", "pyarrow", "requests", "pyyaml"], check=True)

from importlib.metadata import version
print({p: version(p) for p in ("citecurv", "pyalex", "pandas", "pyarrow", "requests", "pyyaml")})

In [ ]:
# ---- Drive, API key (Colab Secrets), config ------------------------------------
import json, os, pathlib
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["OPENALEX_API_KEY"] = userdata.get("OPENALEX_API_KEY")   # never print or save it

from citecurv.config import load_config
cfg = load_config("/content/repo/configs/default.yaml")   # configs/ is not shipped inside the package
assert cfg["data"]["page_size"] <= 100, "per_page above 100 is deprecated by OpenAlex"

BASE = pathlib.Path(DRIVE_ROOT) / "citecurv"   # shared: labels/, S1.3 files
RUN_DIR = BASE / RUN                          # this run's artifacts (contract file names)
(BASE / "labels").mkdir(parents=True, exist_ok=True)
print("run folder:", RUN_DIR)

In [ ]:
# ---- credits left today (re-run on repeated 429s: too fast vs daily budget spent) ----
import requests

r = requests.get("https://api.openalex.org/rate-limit",
                 params={"api_key": os.environ["OPENALEX_API_KEY"]}, timeout=30)
print(r.status_code, (r.json() if r.ok else {}).get("rate_limit"))   # credits_remaining 0 = wait for midnight UTC

In [ ]:
# ---- frozen Retraction Watch snapshot: fetch the pinned commit, verify SHA256 ----
import hashlib
from citecurv.data.rwdb import download_snapshot, load_labels

counts = json.loads(pathlib.Path("/content/repo/data/labels/rwdb_counts.json").read_text())
rwdb_csv = BASE / "labels" / counts["snapshot"]["filename"]
if not rwdb_csv.exists():
    url = f"https://gitlab.com/crossref/retraction-watch-data/-/raw/{RWDB_COMMIT}/retraction_watch.csv"
    download_snapshot(rwdb_csv, url=url)
sha = hashlib.sha256(rwdb_csv.read_bytes()).hexdigest()
assert sha == counts["snapshot"]["sha256"], f"RWDB snapshot hash mismatch: {sha}"

labels = load_labels(rwdb_csv, families=LABEL_FAMILIES)
labels = labels[labels["retraction_nature"].isin(LABEL_NATURES)]
print("labeled papers (distinct DOIs):", labels["doi_norm"].nunique())

## S1.3: choose the slice

1. Look up every labeled DOI in OpenAlex (about 290 calls) to see which topics hold the labels.
   `dois_found / dois_queried` is also the strict DOI found-rate for S1.2.
2. For each shortlist candidate: size its subfield set per topic, add topics in order of label
   density until `TARGET_WORKS`, then measure the narrowed slice (works, references, labels).
3. Set `CHOSEN` and re-run step 2's cell to write `slice.yaml`; copy it to `configs/slice.yaml`.

In [ ]:
if STAGE == "S1.3":
    from citecurv.data import slice_query as sq

    lookup_path = BASE / "labeled_works_openalex.parquet"
    if lookup_path.exists():
        import pandas as pd
        lookup = pd.read_parquet(lookup_path)
    else:
        lookup, lookup_stats = sq.lookup_labeled_works(sorted(set(labels["doi_norm"].dropna())), config=cfg)
        lookup.to_parquet(lookup_path, index=False)
        print(lookup_stats)
    print(lookup.groupby("subfield_id")["doi_norm"].nunique().sort_values(ascending=False).head(15))

In [ ]:
if STAGE == "S1.3":
    import pandas as pd

    measured, narrowed = {}, {}
    for cand in sq.SHORTLIST:
        if cand["openalex"].get("needs_resolution"):
            print("skip (resolve source IDs first):", cand["name"]); continue
        broad = sq.measure_slice(cand, dois=None, config=cfg)          # sizes only, 6 calls
        pick = sq.pick_topics(broad["works_by_topic"], sq.labeled_by_topic(lookup, cand["years"]),
                              target_works=TARGET_WORKS)
        if not pick["topics"]:
            print("no labeled topics:", cand["name"]); continue
        nc = sq.topic_candidate(cand, pick["topics"])
        lo, hi = cand["years"]
        yr = pd.to_numeric(lookup["publication_year"], errors="coerce").astype(float)
        in_set = lookup["topic_id"].isin(pick["topics"]) & yr.between(lo - 1, hi + 1)
        measured[nc["name"]] = sq.measure_slice(nc, dois=sorted(set(lookup.loc[in_set, "doi_norm"])), config=cfg)
        measured[nc["name"]]["broad_works_total"] = broad["works_total"]
        narrowed[nc["name"]] = nc
    (BASE / "slice_measurements.json").write_text(json.dumps(
        {"measured": measured, "candidates": narrowed}, indent=2, default=str))
    show = ["broad_works_total", "works_total", "with_refs_share", "edges_est", "labeled_total",
            "labeled_test", "pull_cost_usd_est", "passes"]
    print(pd.DataFrame(measured).T[show])
    if CHOSEN:
        sq.write_slice_yaml(BASE / "slice.yaml", narrowed[CHOSEN], measured[CHOSEN],
                            rwdb_snapshot=counts["snapshot"])
        print(open(BASE / "slice.yaml").read())   # copy into configs/slice.yaml and commit

## S1.2 pilot / S4.0 full slice: pull, join, write parquet

Pulls run one publication year at a time into `RUN_DIR/year=YYYY/`. Each year's pull is resumable: its
cursor and counts are saved to `year=YYYY/pull_state.json` after every page, so a rerun with the same
parameters resumes inside the year, and a finished year returns its counts without any request. If the
daily OpenAlex budget runs out, the cell prints the reset time and stops; rerun the notebook after the
reset. Cursor paging needs the key.

In [ ]:
if STAGE in ("S1.2", "S4.0"):
    from citecurv.data import slice_query as sq
    from citecurv.data.openalex import BudgetExhausted, pull_works

    if STAGE == "S1.2":
        cand = next(c for c in sq.SHORTLIST if c["name"] == PILOT_CANDIDATE)
        buckets = {PILOT_YEAR: {**sq.slice_filter(cand), "publication_year": str(PILOT_YEAR)}}
        max_works = PILOT_MAX_WORKS
    else:
        import yaml
        sl = yaml.safe_load(open("/content/repo/configs/slice.yaml"))["slice"]
        lo, hi = sl["years"]
        buckets = {y: {**sl["openalex_filter"], "publication_year": str(y)} for y in range(lo, hi + 1)}
        max_works = None

    pull_stats = {}
    for year, filters in buckets.items():
        out = RUN_DIR / f"year={year}"
        state = out / "pull_state.json"   # one state file per year: a state is tied to its filters
        try:
            s = pull_works(filters, out, max_works=max_works, config=cfg, state_path=state)
        except BudgetExhausted as e:
            print(e)   # reset time (UTC) and how many pages of this year are saved
            print(f"Resume: after the reset, rerun this notebook top to bottom with the same TAG, STAGE and RUN. "
                  f"Finished years return their counts at once and year {year} continues from {state}.")
            raise
        pull_stats[year] = vars(s)
        print(year, s)

In [ ]:
if STAGE in ("S1.2", "S4.0"):
    import pandas as pd
    from citecurv.data.events import build_events, event_labels_from_rwdb, write_events
    from citecurv.data.rwdb import join_labels

    works = pd.concat([pd.read_parquet(p) for p in sorted(RUN_DIR.glob("year=*/works.parquet"))],
                      ignore_index=True).drop_duplicates("work_id")
    edges = pd.concat([pd.read_parquet(p) for p in sorted(RUN_DIR.glob("year=*/edges.parquet"))],
                      ignore_index=True)
    works.to_parquet(RUN_DIR / "works.parquet", index=False)
    edges.to_parquet(RUN_DIR / "edges.parquet", index=False)

    # event_id is defined once, in code: sort by (t, src_id, dst_id), number 0..n-1, repeated pairs dropped
    events = build_events(edges)
    event_counts = dict(events.attrs)   # n_edges_in, n_duplicates_dropped, n_self_loops
    write_events(events, RUN_DIR / "events.parquet")
    print(f"{len(events):,} events", event_counts)

    _, join_strict = join_labels(works, labels, mode="strict")
    _, join_loose = join_labels(works, labels, mode="loose")
    print("strict:", join_strict)
    print("loose: ", join_loose)

    labels_out = (labels.drop_duplicates("doi_norm")
                  .assign(reason_family="ref_attr", rwdb_snapshot_date=counts["snapshot"]["date"])
                  [["doi_norm", "reason_family", "retraction_date", "rwdb_snapshot_date"]])
    labels_out.to_parquet(RUN_DIR / "labels.parquet", index=False)

    # Event labels (L12, PROVISIONAL(G3)): positive when the citing or the cited paper is labelled. Cited works
    # outside the slice have no DOI in works.parquet; the S1.3 DOI lookup supplies the ids of labelled ones.
    id_doi = works[["work_id", "doi_norm"]]
    lookup_path = BASE / "labeled_works_openalex.parquet"
    if lookup_path.exists():
        id_doi = pd.concat([id_doi, pd.read_parquet(lookup_path)[["work_id", "doi_norm"]].dropna()],
                           ignore_index=True)
    else:
        print("WARNING: no S1.3 lookup; cited works outside the slice can never be positive")
    event_labels = event_labels_from_rwdb(events, id_doi, labels, mode=LABEL_JOIN_MODE)
    event_label_stats = event_labels.attrs["stats"]
    event_labels.attrs = {}
    event_labels.to_parquet(RUN_DIR / "event_labels.parquet", index=False)
    print("event labels:", event_label_stats)

    join_stats = {"stage": STAGE, "run": RUN, "filters": {str(k): v for k, v in buckets.items()},
                  "pull": {str(k): v for k, v in pull_stats.items()},
                  "works": len(works), "edges": len(edges), "events": len(events), "event_counts": event_counts,
                  "join_strict": join_strict, "join_loose": join_loose,
                  "event_labels": {"mode": LABEL_JOIN_MODE, **event_label_stats},
                  "label_families": LABEL_FAMILIES, "label_natures": LABEL_NATURES,
                  "rwdb_snapshot": counts["snapshot"]}
    (RUN_DIR / "join_stats.json").write_text(json.dumps(join_stats, indent=2, default=str))
    print(json.dumps(join_stats, indent=2, default=str))   # counts only: copy into results/ and PROGRESS.md

In [ ]:
if STAGE == "S4.0":
    import datetime

    def sha256(p):
        return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

    expected = sl["expected"]
    manifest = {
        "pull_date": datetime.date.today().isoformat(),
        "slice": sl["name"], "filters": sl["openalex_filter"], "years": sl["years"],
        "openalex": {**sl.get("openalex", {}), "pull_date": datetime.date.today().isoformat()},
        "counts": {"works": len(works), "edges": len(edges), "events": len(events),
                   "labeled_works_strict": join_strict["labeled_works"],
                   "positive_events": event_label_stats["positive_events"]},
        "expected": expected,
        "works_within_20pct": abs(len(works) - expected["works"]) <= 0.2 * expected["works"],
        "sha256": {n: sha256(RUN_DIR / n) for n in ("works.parquet", "edges.parquet", "events.parquet",
                                                "labels.parquet", "event_labels.parquet")},
        "rwdb_snapshot": counts["snapshot"],
    }
    (RUN_DIR / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    print(json.dumps(manifest, indent=2, default=str))   # commit this file (task S4.0)